In [1]:
# imports

import csv
import re
import logging
import string
import json
from enum import Enum
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import tifffile

import typing
from numpy.typing import NDArray

In [2]:
# slow imports
from adaptive_polish.strategy import AdaptivePolishMillingStrategy
from adaptive_polish.config import AdaptivePolishMillingConfig
from adaptive_polish.exceptions import StopEarlyError, StopMillingException

from fibsem.structures import Point

Default configuration default-configuration. Configuration Path: C:\Users\tpr78264\code\adapive_polish_workspace\fibsem\fibsem\config\microscope-configuration.yaml


c:\Users\tpr78264\code\adapive_polish_workspace\adaptive_polish\.venv\lib\site-packages\networkx\utils\backends.py:132: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends = _get_backends("networkx.plugins")
C:\Program Files\Enthought\Python\envs\AutoScript\Lib\site-packages\numexpr\interpreter.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import sys, pkg_resources, importlib.util


In [3]:
class NoImageFound(FileNotFoundError):
    pass


class NoExperimentFound(NoImageFound):
    pass

In [4]:
# Constants
CONTINUE_FILENAME_STRING = "mill"
STOP_FILENAME_STRING = "stop"

base_path = (
    Path.home()
    / "OneDrive - The Rosalind Franklin Institute"
    / "Documents"
    / "test data"
    / "adaptive milling"
)

models = {
    "Gen0": (
        "0",
        base_path
        / "sem_models"
        / "Gen0"
        / "2024-02-24_0013_gis_lamela_crack_pytorch_AUnet.ptchkp",
    ),
    "Gen1 v2 Quality": (
        "1.0q",
        base_path
        / "sem_models"
        / "Gen1"
        / "gen01_quality_1536_v2"
        / "20250228_gen01_quality_1536_v2.pth",
    ),
    "Gen1 v3 Quality": (
        "1.0q",
        base_path
        / "sem_models"
        / "Gen1"
        / "gen01_quality_1536_v3"
        / "gen01_quality_1536_v3.pth",
    ),
    "Gen1 Performance": (
        "1.0p",
        base_path
        / "sem_models"
        / "Gen1"
        / "gen01_performance_768"
        / "20250228_gen01_performance_768.pth",
    ),
    "Gen1 v3 Performance": (
        "1.1p",
        base_path
        / "sem_models"
        / "Gen1"
        / "gen01_performance_768_v3"
        / "gen01_performance_768_v3.pth",
    ),
    "Gen1 v4 Quality": (
        "1.1q",
        base_path
        / "sem_models"
        / "Gen1"
        / "gen01_quality_1536_v4"
        / "gen01_quality_1536_v4.pth",
    ),
    "Gen1 v5 Greyscale": (
        "1.2q",
        base_path
        / "sem_models"
        / "Gen1"
        / "gen01_quality_1536_v5_grayscale"
        / "cryo_sem_epoch_56.pth",
    ),
    "Gen1 v7 FPN": (
        "1.3fpn",
        base_path
        / "sem_models"
        / "Gen1"
        / "gen01_quality_1536_v7_FPN"
        / "gen01_quality_1536_v7_FPN.pth",
    ),
}

In [5]:
def get_array_and_pixel_size(image_path: Path) -> tuple[NDArray[typing.Any], float]:
    with tifffile.TiffFile(image_path) as tif:
        image_array = tif.asarray()
        # TODO: What's going on with the metadata?
        # pixel_size_m = tif.fei_metadata
        return image_array, tif.shaped_metadata[0]["pixel_size"]["x"]

In [6]:
@dataclass
class Metadata:
    pixel_size: Point


@dataclass
class TestImageInfo:
    path: Path
    data: NDArray[typing.Any]
    pixel_size_m: float
    expect_continue_milling: bool
    results: dict[str, typing.Any] = field(default_factory=dict)
    metadata: Metadata = field(init=False)

    def __post_init__(self) -> None:
        self.metadata = Metadata(
            pixel_size=Point(x=self.pixel_size_m, y=self.pixel_size_m)
        )


class StrEnum(str, Enum):
    def __str__(self):
        return str(self.value)


class MillingDecision(StrEnum):
    Mill = "milling should continue"
    StopNoCleanPrediction = "no clean prediction"
    StopNoGISLayer = "no GIS layer detected"
    StopLamellaSize = "lamella too small"
    StopLamellaBounds = "failed to get lamella bounds"
    StopDrift = "lamella centre has drifted too much"
    StopTooThinGISLayer = "minimum GIS thickness below threshold"
    StopCrackFound = "crack with area above threshold found"


class AdaptiveMillingTestResult(StrEnum):
    SUCCESS = "success"
    FAILED_TO_STOP = "failed to stop"
    FAILED_TO_CONTINUE = "failed to continue"
    UNEXPECTED_ERROR = "unexpected error"

In [7]:
def get_step_info(
    directory_path: Path, index: int, step_char: str, image_type: str
) -> TestImageInfo:
    file_stem = f"{index:02d}{step_char}_{image_type}"
    continue_path = directory_path / f"{file_stem}_{CONTINUE_FILENAME_STRING}.tif"
    stop_path = directory_path / f"{file_stem}_{STOP_FILENAME_STRING}.tif"
    if continue_path.is_file():
        image_path = continue_path
        expect_continue = True
    elif stop_path.is_file():
        image_path = stop_path
        expect_continue = False
    else:
        raise NoImageFound(f"No image found for {file_stem}")

    image_array, pixel_size_m = get_array_and_pixel_size(image_path)

    return TestImageInfo(image_path, image_array, pixel_size_m, expect_continue)

In [8]:
def process_experiment_step(
    strategy: AdaptivePolishMillingStrategy,
    milling_cycle: int,
    sem_image_info: TestImageInfo,
    fib_image_info: TestImageInfo,
    plot_dir: Path,
    identifier: str,
) -> tuple[AdaptiveMillingTestResult, str]:
    assert (
        sem_image_info.expect_continue_milling == fib_image_info.expect_continue_milling
    ), "Milling continue or stop does not match between SEM and FIB images"

    try:
        strategy._check_lamella(
            milling_cycle=milling_cycle,
            image_name=identifier,
            fib_image=fib_image_info,
            sem_image=sem_image_info,
            plots_folder=plot_dir,
            results_dict={},
        )
        milling_decision = MillingDecision.Mill
    except StopEarlyError as e:
        e_str = str(e)
        if e_str.startswith("No GIS found"):
            milling_decision = MillingDecision.StopNoGISLayer
        elif e_str.startswith("Lamella found was only"):
            milling_decision = MillingDecision.StopLamellaSize
        elif e_str.startswith("Failed to get lamella bounds"):
            milling_decision = MillingDecision.StopLamellaBounds
        elif e_str.startswith("Total drift"):
            milling_decision = MillingDecision.StopDrift
        else:
            return AdaptiveMillingTestResult.UNEXPECTED_ERROR, e_str
    except StopMillingException as e:
        e_str = str(e)
        if e_str.startswith("Minimum GIS thickness"):
            milling_decision = MillingDecision.StopTooThinGISLayer
        elif e_str.startswith("Crack area"):
            milling_decision = MillingDecision.StopCrackFound
        else:
            return AdaptiveMillingTestResult.UNEXPECTED_ERROR, e_str
    except Exception as e:
        return AdaptiveMillingTestResult.UNEXPECTED_ERROR, str(e)

    continue_milling = milling_decision == MillingDecision.Mill
    if sem_image_info.expect_continue_milling:
        if continue_milling:
            return AdaptiveMillingTestResult.SUCCESS, str(milling_decision)
        else:
            logging.warning(
                "Milling failed to continue for %s (expected continue=%s), decision(s): %s",
                identifier,
                sem_image_info.expect_continue_milling,
                str(milling_decision),
            )
            return AdaptiveMillingTestResult.FAILED_TO_CONTINUE, str(milling_decision)
    else:
        if continue_milling:
            logging.warning(
                "Milling failed to stop for %s (expected continue=%s)",
                identifier,
                sem_image_info.expect_continue_milling,
            )
            return AdaptiveMillingTestResult.FAILED_TO_STOP, str(milling_decision)
        else:
            return AdaptiveMillingTestResult.SUCCESS, str(milling_decision)

In [22]:
def process_experiment(
    strategy: AdaptivePolishMillingStrategy,
    fib_path: Path,
    sem_path: Path,
    plot_dir: Path,
    index: int,
) -> list[tuple[AdaptiveMillingTestResult, str]]:
    outputs: list[tuple[AdaptiveMillingTestResult, str]] = []
    step_characters = string.ascii_uppercase
    for milling_cycle, step_char in enumerate(step_characters, 1):
        try:
            sem_image_info = get_step_info(sem_path, index, step_char, "SEM")
            fib_image_info = get_step_info(fib_path, index, step_char, "FIB")
            outputs.append(
                process_experiment_step(
                    strategy,
                    milling_cycle,
                    sem_image_info,
                    fib_image_info,
                    plot_dir,
                    f"{index:02d}_{step_char}",
                )
            )
        except NoImageFound:
            if step_char == step_characters[0]:
                # Only raise an error if it is the first file in the series
                raise NoExperimentFound(f"No experiment {index} found")
            break
    return outputs

In [10]:
def process_from_csv(
    strategy: AdaptivePolishMillingStrategy,
    csv_path: Path,
    plot_dir: Path,
) -> dict[str, list[tuple[AdaptiveMillingTestResult, str]]]:
    image_regex = re.compile(r"^([\w_\-\. ]+)_img_(\d{3})_(?:SEM|FIB)$", re.I)
    results: dict[str, list[tuple[AdaptiveMillingTestResult, str]]] = {}
    with csv_path.open("r") as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) != 3:
                logging.warning("Invalid row of length %i", len(row))
                continue
            sem_path, fib_path, continue_string = row
            expect_continue = continue_string == CONTINUE_FILENAME_STRING
            sem_path = Path(sem_path)
            fib_path = Path(fib_path)
            m = image_regex.match(sem_path.stem)
            if m is None:
                raise NameError(f"Failed to get experiment info from {sem_path.stem}")
            experiment_name = m.group(1)
            milling_cycle_str = m.group(2)
            if experiment_name not in results:
                results[experiment_name] = []
            sem_image_info = TestImageInfo(
                sem_path,
                *get_array_and_pixel_size(sem_path),
                expect_continue_milling=expect_continue,
            )
            fib_image_info = TestImageInfo(
                fib_path,
                *get_array_and_pixel_size(fib_path),
                expect_continue_milling=expect_continue,
            )

            results[experiment_name].append(
                process_experiment_step(
                    strategy,
                    int(milling_cycle_str),
                    sem_image_info,
                    fib_image_info,
                    plot_dir,
                    identifier=f"{experiment_name}_img_{milling_cycle_str}",
                )
            )
    return results

In [ ]:
model_key = "Gen1 v7 FPN"
model_generation, model_path = models[model_key]
config = AdaptivePolishMillingConfig(
    model_path=model_path,
    model_generation=model_generation,
    window_size_px=10,
    max_crack_area_um2=0.5,
    gis_stop_um=0.25,
    align_sem=False,
)

In [12]:
strategy = AdaptivePolishMillingStrategy(config)
strategy._load_model()

In [13]:
def process_csv_file(
    strategy: AdaptivePolishMillingStrategy,
    csv_path: Path,
) -> tuple[
    dict[str, list[tuple[AdaptiveMillingTestResult, str]]],
    Path,
]:
    output_base_path = base_path / "adaptive_milling_plots" / csv_path.stem / model_key
    assert base_path.is_dir()
    if output_base_path.is_dir():
        logging.warning("%s already exists, data may be overwritten", output_base_path)

    output_base_path.mkdir(exist_ok=True, parents=True)

    plot_dir = output_base_path / "plots"
    plot_dir.mkdir(exist_ok=True)

    results = process_from_csv(strategy, csv_path=csv_path, plot_dir=plot_dir)
    return results, output_base_path

In [14]:
def process_data_directory(
    strategy: AdaptivePolishMillingStrategy,
    data_directory: Path,
) -> tuple[
    dict[int, list[tuple[AdaptiveMillingTestResult, str]]],
    Path,
]:
    output_base_path = data_directory / model_key
    if output_base_path.is_dir():
        logging.warning("%s already exists, data may be overwritten", output_base_path)

    output_base_path.mkdir(exist_ok=True)

    plot_dir = output_base_path / "plots"
    plot_dir.mkdir(exist_ok=True)

    # iterate
    i = 1
    results: dict[int, list[tuple[AdaptiveMillingTestResult, str]]] = {}

    while True:
        try:
            results[i] = process_experiment(
                strategy,
                data_directory / "FIB".lower(),
                data_directory / "SEM".lower(),
                plot_dir,
                i,
            )
            i += 1
        except NoExperimentFound:
            logging.info("Stopping processing as no experiment was found for %i", i)
            break
        except Exception:
            logging.error("Unexpected error", exc_info=True)
            raise
    return results, output_base_path

In [23]:
# Pick whether to run a csv or data path:

# results, output_base_path = process_csv_file(
#     strategy,
#     base_path
#     / "data"
#     / "filelists"
#     / "20241007_adaptive_vs_non-apadaptive_01_combined.csv",
# )
results, output_base_path = process_data_directory(
    strategy,
    base_path
    / "20250610_quick_ON_test"
    / "AutoLamella-2025-06-10-21-08"
    / "02-vast-frog"
    / "adaptive_polish_2025-06-11-02-58-27AM",
)

Value(False)


In [16]:
results_json_path = output_base_path / "results.json"
results_pie_plots_path = output_base_path / "results_pie_plots.png"
results_bar_plots_path = output_base_path / "results_bar_plots.png"

In [17]:
with results_json_path.open("w+") as _:
    json.dump(results, _)

In [18]:
# Can be run to avoid rerunning previous bits:
with results_json_path.open("r") as _:
    results: dict[
        typing.Union[int, str], list[tuple[AdaptiveMillingTestResult, str]]
    ] = json.load(_)

In [19]:
def create_pie_plots(
    results: dict[typing.Union[int, str], list[tuple[AdaptiveMillingTestResult, str]]],
    output_path: Path,
) -> None:
    # Generate stats for plots
    experiment_success: dict[str, int] = {str(_): 0 for _ in AdaptiveMillingTestResult}

    image_success: dict[str, int] = {str(_): 0 for _ in AdaptiveMillingTestResult}
    failures_per_experiment: dict[int, int] = {}
    keeps_failing_to_stop: dict[str, int] = {
        "never stops": 0,
        "stops later": 0,
        "no further images": 0,
    }
    good_decisions: dict[str, int] = {str(_): 0 for _ in MillingDecision}
    bad_decisions: dict[str, int] = {str(_): 0 for _ in MillingDecision}

    total_experiments = len(results.keys())
    total_images = sum((len(_) for _ in results.values()))

    print(f"Total number of experiments: {total_experiments}")
    print(f"Total number of images: {total_images}")

    for experiment_idx, experiment_outputs in results.items():
        # Get experiment success
        experiment_results = tuple(_[0] for _ in experiment_outputs)
        if str(AdaptiveMillingTestResult.FAILED_TO_CONTINUE) in experiment_results:
            experiment_success[str(AdaptiveMillingTestResult.FAILED_TO_CONTINUE)] += 1

        elif AdaptiveMillingTestResult.FAILED_TO_STOP in experiment_results:
            experiment_success[str(AdaptiveMillingTestResult.FAILED_TO_STOP)] += 1

            # Check if failing continues after an initial FAILED_TO_STOP
            first_failed_to_stop = experiment_results.index(
                str(AdaptiveMillingTestResult.FAILED_TO_STOP)
            )
            if first_failed_to_stop != len(experiment_results) - 1:
                if len(set(experiment_results[first_failed_to_stop:])) == 1:
                    keeps_failing_to_stop["never stops"] += 1
                else:
                    keeps_failing_to_stop["stops later"] += 1
            else:
                keeps_failing_to_stop["no further images"] += 1

        else:
            experiment_success[str(AdaptiveMillingTestResult.SUCCESS)] += 1

        # Get image success
        failure_count = 0
        for outputs in experiment_outputs:
            image_success[str(outputs[0])] += 1
            if outputs[0] == AdaptiveMillingTestResult.SUCCESS:
                key = outputs[1]
                good_decisions[key] = good_decisions.get(key, 0) + 1
            else:
                failure_count += 1
                key = outputs[1]
                bad_decisions[key] = bad_decisions.get(key, 0) + 1

        # Count how many times failed experiments failed
        failures_per_experiment[failure_count] = (
            failures_per_experiment.get(failure_count, 0) + 1
        )

    fig, axes = plt.subplots(2, 3, figsize=(20, 14), dpi=300)

    fig.suptitle("Results of SEM GIS detection\n\n", size="x-large")

    axes[0, 0].pie(
        experiment_success.values(),
        labels=tuple(str(_) if _ > 0 else "" for _ in experiment_success.values()),
    )
    axes[0, 0].legend(experiment_success.keys(), loc="best")
    axes[0, 0].set_title("Experiment results")

    axes[0, 1].pie(
        failures_per_experiment.values(),
        labels=failures_per_experiment.values(),
    )
    axes[0, 1].legend(
        tuple(f"Failures: {_}" for _ in failures_per_experiment.keys()), loc="best"
    )
    axes[0, 1].set_title("Number of failures per experiment")

    axes[0, 2].pie(
        keeps_failing_to_stop.values(), labels=keeps_failing_to_stop.values()
    )
    axes[0, 2].legend(keeps_failing_to_stop.keys(), loc="best")
    axes[0, 2].set_title("If an experiment fails to stop, what happens next?")

    axes[1, 0].pie(
        image_success.values(),
        labels=tuple(str(_) if _ > 0 else "" for _ in image_success.values()),
    )
    axes[1, 0].legend(image_success.keys(), loc="best")
    axes[1, 0].set_title("Image results")

    axes[1, 1].pie(
        good_decisions.values(),
        labels=tuple(str(_) if _ > 0 else "" for _ in good_decisions.values()),
        shadow=False,
    )
    axes[1, 1].legend(good_decisions.keys(), loc="best")
    axes[1, 1].set_title("Good image decision reasons")

    axes[1, 2].pie(
        bad_decisions.values(),
        labels=bad_decisions.values(),
    )
    axes[1, 2].legend(bad_decisions.keys(), loc="best")
    axes[1, 2].set_title("Bad image decision reasons")

    fig.tight_layout()
    fig.savefig(output_path)
    fig.show()

In [20]:
def create_bar_plots(
    results: dict[typing.Union[int, str], list[tuple[AdaptiveMillingTestResult, str]]],
    output_path: Path,
) -> None:
    # Generate stats for plots
    experiment_success: dict[str, int] = {str(_): 0 for _ in AdaptiveMillingTestResult}

    image_success: dict[str, int] = {str(_): 0 for _ in AdaptiveMillingTestResult}
    failures_per_experiment: dict[int, int] = {}
    keeps_failing_to_stop: dict[str, int] = {
        "never stops": 0,
        "stops later": 0,
        "no further images": 0,
    }
    good_decisions: dict[str, int] = {str(_): 0 for _ in MillingDecision}
    bad_decisions: dict[str, int] = {str(_): 0 for _ in MillingDecision}

    total_experiments = len(results.keys())
    total_images = sum((len(_) for _ in results.values()))

    print(f"Total number of experiments: {total_experiments}")
    print(f"Total number of images: {total_images}")

    for experiment_idx, experiment_outputs in results.items():
        # Get experiment success
        experiment_results = tuple(_[0] for _ in experiment_outputs)
        if str(AdaptiveMillingTestResult.FAILED_TO_CONTINUE) in experiment_results:
            experiment_success[str(AdaptiveMillingTestResult.FAILED_TO_CONTINUE)] += 1

        elif AdaptiveMillingTestResult.FAILED_TO_STOP in experiment_results:
            experiment_success[str(AdaptiveMillingTestResult.FAILED_TO_STOP)] += 1

            # Check if failing continues after an initial FAILED_TO_STOP
            first_failed_to_stop = experiment_results.index(
                str(AdaptiveMillingTestResult.FAILED_TO_STOP)
            )
            if first_failed_to_stop != len(experiment_results) - 1:
                if len(set(experiment_results[first_failed_to_stop:])) == 1:
                    keeps_failing_to_stop["never stops"] += 1
                else:
                    keeps_failing_to_stop["stops later"] += 1
            else:
                keeps_failing_to_stop["no further images"] += 1

        else:
            experiment_success[str(AdaptiveMillingTestResult.SUCCESS)] += 1

        # Get image success
        failure_count = 0
        for outputs in experiment_outputs:
            image_success[str(outputs[0])] += 1
            if outputs[0] == AdaptiveMillingTestResult.SUCCESS:
                key = outputs[1]
                good_decisions[key] = good_decisions.get(key, 0) + 1
            else:
                failure_count += 1
                key = outputs[1]
                bad_decisions[key] = bad_decisions.get(key, 0) + 1

        # Count how many times failed experiments failed
        failures_per_experiment[failure_count] = (
            failures_per_experiment.get(failure_count, 0) + 1
        )

    fig, axes = plt.subplots(2, 3, figsize=(20, 14), dpi=300)

    fig.suptitle("Results of SEM GIS detection\n\n", size="x-large")

    _ = axes[0, 0].barh(
        experiment_success.keys(),
        experiment_success.values(),
        align="center",
    )
    axes[0, 0].bar_label(_, padding=2)
    axes[0, 0].set_xlim(0, total_experiments)
    axes[0, 0].set_title("Experiment results")

    _ = axes[0, 1].barh(
        tuple(str(_) for _ in failures_per_experiment.keys()),
        failures_per_experiment.values(),
        align="center",
    )
    axes[0, 1].bar_label(_, padding=2)
    axes[0, 1].set_xlim(0, total_experiments)
    axes[0, 1].set_title("Number of failures per experiment")

    _ = axes[0, 2].barh(
        keeps_failing_to_stop.keys(), keeps_failing_to_stop.values(), align="center"
    )
    axes[0, 2].bar_label(_, padding=2)
    axes[0, 2].set_xlim(0, sum(keeps_failing_to_stop.values()))
    axes[0, 2].set_title("If an experiment fails to stop, what happens next?")

    _ = axes[1, 0].barh(image_success.keys(), image_success.values(), align="center")
    axes[1, 0].bar_label(_, padding=2)
    axes[1, 0].set_xlim(0, total_images)
    axes[1, 0].set_title("Image results")

    _ = axes[1, 1].barh(
        tuple(str(_) for _ in good_decisions.keys()),
        good_decisions.values(),
        align="center",
    )
    axes[1, 1].bar_label(_, padding=2)
    axes[1, 1].set_xlim(0, sum(good_decisions.values()))
    axes[1, 1].set_title("Good image decision reasons")

    _ = axes[1, 2].barh(
        tuple(str(_) for _ in bad_decisions.keys()),
        bad_decisions.values(),
        align="center",
    )
    axes[1, 2].bar_label(_, padding=2)
    axes[1, 2].set_xlim(0, sum(bad_decisions.values()))
    axes[1, 2].set_title("Bad image decision reasons")

    fig.tight_layout()
    fig.savefig(output_path)
    fig.show()

In [21]:
create_pie_plots(results, output_path=results_pie_plots_path)

Total number of experiments: 0
Total number of images: 0


c:\Users\tpr78264\code\adapive_polish_workspace\adaptive_polish\.venv\lib\site-packages\matplotlib\axes\_axes.py:3290: RuntimeWarning: invalid value encountered in divide
  x = x / sx


ValueError: cannot convert float NaN to integer

ValueError: need at least one array to concatenate

<Figure size 6000x4200 with 6 Axes>

In [ ]:
create_bar_plots(results, output_path=results_bar_plots_path)